# Aula 03 — Laboratório
## Previsão e causalidade sobre a mesma base

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Uma pesquisa de rendimentos com 4.000 trabalhadores. Você vai fazer **duas perguntas
diferentes** aos mesmos dados:

1. *qual é o retorno de um ano a mais de estudo?* — a pergunta da Econometria;
2. *quanto ganha esta pessoa?* — a pergunta do Machine Learning.

E vai descobrir, no fim, que o modelo que responde melhor à segunda é justamente
o que responde pior à primeira.

> A base é **simulada**. É a única forma de, no fim do dia, comparar o coeficiente
> que você estimou com o efeito que de fato existe no mundo.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver
`___`, substitua. Rode as células na ordem.

## Bloco 0 — ambiente, base e ferramentas

In [ ]:
# Bloco 0 — rode esta célula e confira a saída.
import sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 20)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Azul = treino / dentro da amostra.
# Laranja = teste / fora da amostra. Verde = o terceiro modelo. Bordo = verdade.
AZUL, LARANJA, VERDE, BORDO = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B"

# ---------------------------------------------------------------- PLANO B --
# statsmodels e scikit-learn sao confortaveis, mas nada aqui depende deles.
# Se faltarem, o notebook segue com implementacoes proprias e avisa.
try:
    import statsmodels.api as sm
    TEM_SM = True
except Exception:
    sm = None
    TEM_SM = False

try:
    from sklearn.neighbors import KNeighborsRegressor
    TEM_SK = True
except Exception:
    TEM_SK = False

print("Python     :", sys.version.split()[0], "|", platform.system())
print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("statsmodels:", "ok" if TEM_SM else "AUSENTE - usando a regressao propria")
print("sklearn    :", "ok" if TEM_SK else "AUSENTE - usando o KNN proprio")
print("\nAmbiente pronto.")

In [ ]:
# Bloco 0 — a base do dia.
#
#  +--------------------------------------------------------------------+
#  |  ATENCAO: esta celula CONSTROI o mundo, entao ela contem a resposta |
#  |  do dia. Rode-a e siga em frente. Volte a LER o codigo so no        |
#  |  Bloco 5 — ler agora estraga o exercicio inteiro.                   |
#  +--------------------------------------------------------------------+
#
# Uma pesquisa domiciliar ficticia de rendimentos: 4.000 trabalhadores,
# estrutura de uma PNAD reduzida.

EFEITO_DIRETO = 0.06
EFEITO_HABILIDADE = 0.26
EFEITO_EXPERIENCIA = 0.03
EFEITO_EXPERIENCIA2 = -0.0005
EFEITO_CARGO = 0.8
RUIDO = 0.33
CORR_ORIGEM_HAB = 0.92
SETORES = ('Serviços', 'Indústria', 'Comércio', 'Público')
EFEITO_SETOR = {'Serviços': 0.0, 'Indústria': 0.1, 'Comércio': -0.08, 'Público': 0.18}
REGIOES = ('Nordeste', 'Sudeste', 'Sul', 'Norte', 'Centro-Oeste')
EFEITO_REGIAO = {'Nordeste': 0.0, 'Sudeste': 0.2, 'Sul': 0.14, 'Norte': 0.02, 'Centro-Oeste': 0.1}
BASE_LOG = 7.9
RUIDO_PILOTO = 0.07
N_PILOTO = 25
CENTRO = 20.0
ESCALA = 20.0


def gerar_base(n=4000, semente=42, anos_a_mais=0.0):
    """Uma pesquisa domiciliar fictícia de rendimentos.

    Estrutura de uma PNAD reduzida: uma linha por trabalhador, salário mensal,
    anos de estudo, experiência, origem familiar, setor, região e ocupação de
    chefia. A HABILIDADE existe no mundo e entra no salário, mas NÃO é
    observada — é ela que faz a regressão ingênua mentir.

    `anos_a_mais` faz a intervenção: dá a todo mundo esse tanto de estudo a
    mais, mantendo habilidade, sorteios e ruídos idênticos. É assim que o
    efeito causal verdadeiro é medido, e não lido de um parâmetro.
    """
    rng = np.random.default_rng(semente)

    # habilidade: existe, afeta estudo e salário, e ninguém mede
    hab = rng.normal(0, 1, n)

    # origem familiar (escolaridade dos pais, padronizada): observada, e
    # correlacionada com a habilidade — por isso serve de proxy imperfeita
    origem = (CORR_ORIGEM_HAB * hab
              + np.sqrt(1 - CORR_ORIGEM_HAB ** 2) * rng.normal(0, 1, n))

    # anos de estudo: quem tem mais habilidade e melhor origem estuda mais
    estudo = 9.0 + 1.00 * origem + 1.00 * hab + rng.normal(0, 1.8, n)
    estudo = np.clip(np.round(estudo), 4, 20) + anos_a_mais

    # experiência: anos de trabalho, independente do estudo neste mundo
    experiencia = np.clip(np.round(rng.normal(18, 9, n)), 0, 40)

    setor = rng.choice(SETORES, n, p=[0.45, 0.20, 0.22, 0.13])
    regiao = rng.choice(REGIOES, n, p=[0.30, 0.34, 0.14, 0.08, 0.14])

    # cargo de chefia: CONSEQUÊNCIA do estudo e da habilidade. Controlar por
    # ele fecha parte do caminho pelo qual o estudo aumenta o salário.
    indice = 0.85 * (estudo - 11) / 3 + 0.70 * hab + 0.03 * experiencia
    cargo = (indice + rng.normal(0, 1, n) > 1.10).astype(int)

    log_salario = (
        6.40
        + EFEITO_DIRETO * estudo
        + EFEITO_EXPERIENCIA * experiencia
        + EFEITO_EXPERIENCIA2 * experiencia ** 2
        + EFEITO_HABILIDADE * hab
        + EFEITO_CARGO * cargo
        + np.array([EFEITO_SETOR[s] for s in setor])
        + np.array([EFEITO_REGIAO[r] for r in regiao])
        + rng.normal(0, RUIDO, n)
    )

    return {
        "salario": np.round(np.exp(log_salario), 2),
        "log_salario": log_salario,
        "estudo": estudo,
        "experiencia": experiencia,
        "origem": np.round(origem, 3),
        "setor": setor,
        "regiao": regiao,
        "cargo": cargo,
        "_habilidade": hab,          # não entregar ao aluno
    }


dados = pd.DataFrame({k: v for k, v in gerar_base().items()
                      if not k.startswith("_")})
print(f"{len(dados)} trabalhadores, {dados.shape[1]} colunas.")
dados.head(8)

In [ ]:
# Bloco 0 — as ferramentas do dia. Leia, nao precisa alterar.
#
# `ols` faz o que statsmodels faria: coeficientes, R2 e erros-padrao. Serve de
# Plano B e, principalmente, deixa claro que nao ha magica nenhuma ali dentro.

def ols(X, y, nomes=None):
    """Mínimos quadrados com constante. Devolve coeficientes, R² e erros-padrão."""
    X = np.column_stack([np.ones(len(y)), np.asarray(X, dtype=float).reshape(len(y), -1)])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    sq = float(resid @ resid)
    r2 = 1 - sq / float(((y - y.mean()) ** 2).sum())
    gl = len(y) - X.shape[1]
    var = sq / gl * np.linalg.pinv(X.T @ X)
    ep = np.sqrt(np.diag(var))
    if nomes is None:
        nomes = ["const"] + [f"x{i}" for i in range(1, X.shape[1])]
    return {"coef": dict(zip(nomes, beta)), "r2": r2,
            "ep": dict(zip(nomes, ep)), "n": len(y)}

def dummies(valores, categorias):
    """Uma coluna 0/1 por categoria, menos a primeira (a de referência)."""
    return np.column_stack([(valores == c).astype(float) for c in categorias[1:]])

def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))


def regressao(y, X, nomes):
    """Roda no statsmodels se houver; senao, na funcao acima. Mesma saida."""
    if TEM_SM:
        A = sm.add_constant(np.asarray(X, dtype=float))
        r = sm.OLS(np.asarray(y, dtype=float), A).fit()
        return {"coef": dict(zip(nomes, r.params)), "r2": float(r.rsquared),
                "ep": dict(zip(nomes, r.bse)), "n": int(r.nobs)}
    return ols(X, y, nomes)


print("Ferramentas prontas.")

In [ ]:
# Bloco 0 — o primeiro olhar. COMPLETE.
#
# Responda, com codigo: quantos anos de estudo em media? qual a mediana do
# salario? que fracao da amostra ocupa cargo de chefia?

# COMPLETE: descritivas das variaveis numericas
___

# COMPLETE: fracao com cargo de chefia
___

## Bloco 1 — a primeira regressão

A pergunta é: **quanto vale um ano a mais de estudo?**

In [ ]:
# Bloco 1 — a regressao ingenua. Resolvida, para servir de modelo.
r1 = regressao(dados["log_salario"], dados[["estudo"]].to_numpy(),
               ["const", "estudo"])
print(f"beta do estudo : {r1['coef']['estudo']:.4f}")
print(f"erro-padrao    : {r1['ep']['estudo']:.4f}")
print(f"R2             : {r1['r2']:.4f}")
print(f"\nLeitura: cada ano a mais de estudo aparece associado a "
      f"{100 * r1['coef']['estudo']:.1f}% a mais de salario.")

In [ ]:
# Bloco 1 — a leitura economica. COMPLETE.
#
# 1. O coeficiente e enorme e o erro-padrao e minusculo. Isso prova que o
#    efeito e grande? Escreva sua resposta no comentario abaixo.
# 2. Que variavel, se existisse na base, voce colocaria nesta regressao para
#    desconfiar do numero acima?

# RESPOSTA 1: ___
# RESPOSTA 2: ___

# COMPLETE: o grafico da nuvem com a reta ajustada
___

## Bloco 2 — ajuste não é identificação 

Quatro especificações, cada uma acrescentando variáveis à anterior. Anote
**β do estudo** e **R²** de cada uma.

In [ ]:
# Bloco 2 — as quatro especificacoes. COMPLETE.
d_set = dummies(dados["setor"].to_numpy(), SETORES)
d_reg = dummies(dados["regiao"].to_numpy(), REGIOES)
S = dados["estudo"].to_numpy()
X = dados["experiencia"].to_numpy()
y = dados["log_salario"].to_numpy()

modelos = {}
modelos["(1) so estudo"] = regressao(y, np.column_stack([S]), ["const", "estudo"])

# COMPLETE: (2) acrescente experiencia, experiencia ao quadrado, setor e regiao
modelos["(2) + experiencia, setor e regiao"] = ___

# COMPLETE: (3) acrescente a origem familiar
modelos["(3) + origem familiar"] = ___

# COMPLETE: (4) acrescente o cargo de chefia
modelos["(4) + cargo de chefia"] = ___

tabela = pd.DataFrame({
    "beta_estudo": {k: v["coef"]["estudo"] for k, v in modelos.items()},
    "R2": {k: v["r2"] for k, v in modelos.items()},
})
tabela

In [ ]:
# Bloco 2 — o desenho que resume o bloco. COMPLETE.
#
# Dois paineis empilhados: em cima o R2 em barras, embaixo o beta do estudo.
# COMPLETE: desenhe
___

### ✔ CHECKPOINT I

Duas frases escritas:

A especificação **(4)**, com o cargo de chefia, é a de **maior R²**. 
Por que ela é a melhor para **prever** o salário de alguém e a pior para **medir** o retorno do estudo?

## Bloco 3 — a mesma base, agora para prever

Muda a pergunta, muda tudo: o erro passa a ser medido em quem o modelo **não viu**.

In [ ]:
# Bloco 3 — polinomio, com a variavel em escala.
#
# A experiencia vai ate 40. Elevar 40 a decima potencia passa de 10^16: o
# ajuste vira lixo numerico muito antes de virar sobreajuste. Por isso a
# variavel e levada para [-1, 1] ANTES de ajustar. Nada muda no que o modelo
# diz; muda so a aritmetica. Quem esquece disso ve o grau 10 se comportar
# como grau 4 — e o exercicio do dia nao acontece.

def ajusta_polinomio(x, y, grau):
    with np.errstate(all='ignore'):
        return np.polyfit(_z(x), y, grau)

def preve_polinomio(coef, x):
    return np.polyval(coef, _z(x))


def _z(x):
    return (np.asarray(x, dtype=float) - CENTRO) / ESCALA


print(f"40 elevado a 10 = {40.0 ** 10:.3e}")
print("Sem escala, a matriz do ajuste mistura colunas de ordem 1 com colunas")
print("de ordem 10^16, e o resultado e ruido de arredondamento.")
print(f"Com a variavel em [-1, 1], a maior coluna vale {1.0:.0f}. Pronto.")

In [ ]:
# Bloco 3 — o recorte do dia.
#
# O mesmo mundo, olhado onde SO a experiencia varia: 25 trabalhadores do mesmo
# setor, mesma regiao e mesma faixa de estudo. E o recorte em que a relacao
# entre experiencia e salario aparece limpa — e em que sobreajustar e facil.

def curva_verdadeira(x):
    """Log do salário esperado em função da experiência: sobe e depois cede."""
    return BASE_LOG + EFEITO_EXPERIENCIA * np.asarray(x, dtype=float) \
        + EFEITO_EXPERIENCIA2 * np.asarray(x, dtype=float) ** 2

def amostra_piloto(n=N_PILOTO, semente=5, ruido=RUIDO_PILOTO):
    rng = np.random.default_rng(semente)
    x = np.sort(rng.uniform(0, 40, n))
    return x, curva_verdadeira(x) + rng.normal(0, ruido, n)

def amostra_teste(n=4000, semente=99, ruido=RUIDO_PILOTO):
    rng = np.random.default_rng(semente)
    x = np.sort(rng.uniform(0, 40, n))
    return x, curva_verdadeira(x) + rng.normal(0, ruido, n)


x_treino, y_treino = amostra_piloto()      # 25 pessoas
x_teste, y_teste = amostra_teste()         # 4.000 pessoas que o modelo nao vera

fig, ax = plt.subplots()
ax.plot(x_teste, y_teste, "o", ms=4, mfc="none", mec=LARANJA, alpha=0.25)
ax.plot(x_treino, y_treino, "o", ms=9, color=AZUL, mec="white", mew=1.5)
ax.set_xlabel("Anos de experiencia"); ax.set_ylabel("Log do salario")
ax.set_title(f"{len(x_treino)} no treino (azul), {len(x_teste)} no teste (laranja)")
plt.show()

In [ ]:
# Bloco 3 — a curva em U. COMPLETE.
#
# Para cada grau de 0 a 10: ajuste NO TREINO, meca o RMSE no treino e no teste.

graus = list(range(0, 11))
erro_treino, erro_teste = [], []

for g in graus:
    # COMPLETE: ajuste o polinomio de grau g no treino
    coef = ___
    # COMPLETE: guarde o RMSE no treino e no teste
    erro_treino.append(___)
    erro_teste.append(___)

resumo = pd.DataFrame({"grau": graus, "treino": erro_treino, "teste": erro_teste})
print(resumo.round(4))
print("\nmelhor grau pelo teste:", graus[int(np.argmin(erro_teste))])

In [ ]:
# Bloco 3 — o grafico das duas curvas. COMPLETE.
# COMPLETE: erro de treino em azul, erro de teste em laranja, grau no eixo x
___

In [ ]:
# Bloco 3 — os tres retratos. COMPLETE.
#
# Desenhe, lado a lado, o ajuste de grau 1, do melhor grau e de grau 10 sobre
# a nuvem de treino.
# COMPLETE:
___

## Bloco 4 — um modelo sem coeficiente nenhum

O KNN prevê pela **média dos k vizinhos mais próximos**. Não há parâmetro para interpretar, não há reta, não há R² comparável. Só previsão.

In [ ]:
# Bloco 4 — o KNN em dez linhas. COMPLETE.
def meu_knn(x_treino, y_treino, x_novo, k):
    """Para cada valor em x_novo, a media dos k vizinhos mais proximos."""
    xt = np.asarray(x_treino, dtype=float)
    yt = np.asarray(y_treino, dtype=float)
    saida = np.empty(len(np.atleast_1d(x_novo)))
    for i, xv in enumerate(np.atleast_1d(x_novo)):
        # COMPLETE: indices dos k vizinhos mais proximos de xv
        vizinhos = ___
        # COMPLETE: a previsao e a media do y desses vizinhos
        saida[i] = ___
    return saida


# teste rapido: com k = 1 o erro de TREINO tem de ser exatamente zero
print("erro de treino com k=1:",
      rmse(y_treino, meu_knn(x_treino, y_treino, x_treino, 1)))

In [ ]:
# Bloco 4 — varra o k. COMPLETE.
#
# Use uma amostra de treino MAIOR: com 25 pontos nao ha vizinhos suficientes
# para k grande fazer sentido.
x_tr2, y_tr2 = amostra_piloto(n=200, semente=17)

ks = [1, 2, 3, 5, 8, 12, 20, 35, 60, 100]
# COMPLETE: erro de treino e de teste para cada k
knn_treino = ___
knn_teste = ___

print(pd.DataFrame({"k": ks, "treino": knn_treino, "teste": knn_teste}).round(4))
print("\nmelhor k:", ks[int(np.argmin(knn_teste))])

In [ ]:
# Bloco 4 — quem preve melhor? COMPLETE.
#
# Compare, no MESMO conjunto de teste: o melhor polinomio e o melhor KNN.
# COMPLETE:
___

### ✔ CHECKPOINT II

O KNN não tem coeficiente de escolaridade — não tem coeficiente nenhum.

Em duas frases: que **pergunta econômica** ele consegue responder, e que pergunta
ele não alcança por mais que preveja bem?

## Bloco 5 — validação cruzada e o mundo revelado

Duas coisas: escolher a complexidade **sem gastar o teste**, e depois abrir o mundo para ver quem chegou perto.

In [ ]:
# Bloco 5 — validacao cruzada de 5 dobras. COMPLETE.
#
# Divida o TREINO em 5 dobras. Para cada grau, treine em 4 e meca na que ficou
# de fora, cinco vezes. A media das cinco escolhe o grau. O teste fica intacto.

rng = np.random.default_rng(11)
ordem = rng.permutation(len(x_treino))
dobras = np.array_split(ordem, 5)

erro_vc = {}
for g in graus:
    erros = []
    for d in dobras:
        treino = np.setdiff1d(ordem, d)
        # COMPLETE: ajuste nas 4 dobras e meca na que ficou de fora
        coef = ___
        erros.append(___)
    erro_vc[g] = float(np.mean(erros))

g_vc = min(erro_vc, key=erro_vc.get)
print(pd.Series(erro_vc, name="erro medio").round(4))
print(f"\ngrau escolhido pela validacao cruzada : {g_vc}")
print(f"grau que o teste teria escolhido      : {melhor}")

### O mundo revelado

Agora sim: **volte e leia** a célula que construiu a base. Ela mostra como o salário foi gerado — e permite fazer o que nenhuma base real permite: **intervir**.

In [ ]:
# Bloco 5 — o efeito causal verdadeiro, medido por intervencao.
#
# Gera a MESMA populacao duas vezes, com os mesmos sorteios de habilidade,
# setor, regiao e ruido — e na segunda da um ano de estudo a mais a todos.
# A diferenca media no log do salario e o efeito causal total, incluindo a
# parte que passa por virar chefe. Ele NAO esta escrito em nenhuma linha
# do modelo: e preciso medir.

def efeito_causal_verdadeiro(n=200000, semente=2024):
    """Mede o efeito de UM ano a mais de estudo intervindo no mundo.

    Gera a mesma população duas vezes — com os mesmos sorteios de habilidade,
    setor, região e ruído — e na segunda dá um ano de estudo a mais a todos.
    A diferença média no log do salário é o efeito causal total, incluindo a
    parte que passa por virar chefe. É o número que a turma persegue, e ele
    NÃO está escrito em nenhuma linha do modelo.
    """
    a = gerar_base(n=n, semente=semente, anos_a_mais=0.0)
    b = gerar_base(n=n, semente=semente, anos_a_mais=1.0)
    return float(np.mean(b["log_salario"] - a["log_salario"]))


TAU = efeito_causal_verdadeiro()
print(f"efeito causal verdadeiro de 1 ano de estudo: {TAU:.4f}")
print(f"ou seja, cerca de {100 * TAU:.1f}% a mais de salario.")

In [ ]:
# Bloco 5 — o confronto. COMPLETE.
#
# Para cada uma das quatro especificacoes: o beta estimado, a distancia ate o
# verdadeiro, e o erro de previsao fora da amostra.

# COMPLETE: monte a tabela com beta, |beta - TAU| e R2
veredito = ___
veredito

In [ ]:
# Bloco 5 — a prova das duas reguas. COMPLETE.
#
# Separe a base em 70% treino e 30% teste. Para cada especificacao, meca o RMSE
# fora da amostra. Compare com a coluna distancia_ate_TAU acima.
# COMPLETE:
___

### Responda

> Se um **secretário de educação** lhe pedisse o retorno de um ano a mais de escola,
> qual dos seus modelos você entregaria? E se um **banco** quisesse estimar a renda
> de clientes a partir do cadastro, qual você entregaria? E, por fim: as duas respostas são o mesmo modelo?